# ADA Preview 1

Run the canonical ADA causal language model from a fresh checkout. This CPU
experiment trains on generated integer sequences, checks causal prefix
invariance, verifies interrupted training against uninterrupted training,
and exercises the repository's generation loop.

**Setup.** Use Python 3.11+ with PyTorch 2.6+ and a Python Jupyter kernel.
From the ADA checkout, install `python -m pip install -e ".[neural]" jupyterlab`,
start Jupyter, then run all cells in order. The cells run offline. They
create only synthetic data and temporary local checkpoints. This notebook
requires a checkout containing the atomic checkpoint and prefill changes
introduced on `main` before `b4cf9c3`.

**Interpretation.** This is a correctness fixture. The dataset size,
short context and single host are insufficient for language quality,
throughput rankings, distributed durability or native cache claims.
The separate [FineWeb-Edu pilot](../colab/ADA_Pretraining.ipynb) covers
real-data training under its own dataset and runtime requirements.

In [1]:
from importlib import import_module
import errno
import json
import math
from pathlib import Path
import random
import sys
import tempfile
from unittest.mock import patch

import torch

def find_checkout(start):
    candidates = (start, *start.parents)
    for candidate in candidates:
        if (candidate / "ada/atomic_checkpoint.py").is_file() and (candidate / "pyproject.toml").is_file():
            return candidate
    raise RuntimeError("Open this notebook from inside a current ADA repository checkout.")

checkout = find_checkout(Path.cwd().resolve())
sys.path.insert(0, str(checkout))
from ada import atomic_checkpoint
from ada.neural import ADAConfig, ADAForCausalLM
from ada.pretraining_resume import replay_completed_batches

generation = import_module("18_INFERENCE.inference_engine")
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
seed = 317
vocabulary_size = 19
batch_size = 4
sequence_length = 8
print({"python": sys.version.split()[0], "torch": torch.__version__, "device": "cpu", "seed": seed})

{'python': '3.11.0rc2', 'torch': '2.11.0+cpu', 'device': 'cpu', 'seed': 317}


## Data and model

Each record is a short arithmetic sequence modulo 19. Training and
validation use disjoint starting offsets. This deliberately small task
tests the supervised path and shared transition pattern, with no claim
about out-of-distribution language generalization. Both model variants
receive the same training batches and use the same optimizer settings.

Compare ADA's gated convolution mixer with the explicit Mamba-2 reference
mixer using the same hidden width and decoder depth. The mixer swap changes
parameter count. Report it alongside loss and treat the comparison as a
smoke experiment, independent of a compute-matched benchmark.

In [2]:
def rows(starts):
    offsets = torch.arange(sequence_length + 1)
    starts = torch.as_tensor(starts, dtype=torch.long)
    return (starts[:, None] + offsets[None, :]) % vocabulary_size

train_offsets = torch.arange(15)
val_offsets = torch.arange(15, vocabulary_size)
assert not (set(train_offsets.tolist()) & set(val_offsets.tolist()))
validation = rows(val_offsets)
picker = torch.Generator().manual_seed(seed + 1)
train_batches = [
    rows(train_offsets[torch.randint(len(train_offsets), (batch_size,), generator=picker)])
    for _ in range(32)
]
assert train_batches[0].shape == (batch_size, sequence_length + 1)

def new_pair(backend):
    torch.manual_seed(seed)
    random.seed(seed)
    config = ADAConfig(
        vocab_size=vocabulary_size, dim=16, n_layers=2, n_heads=2,
        attention_rank=4, max_seq_len=16, num_experts=2, num_slots=4,
        expansion_factor=1, mixer_expand=2, mixer_kernel_size=2,
        mixer_backend=backend, mamba2_d_state=4, mamba2_headdim=4,
        mamba2_ngroups=1, mamba2_chunk_size=4,
    )
    model = ADAForCausalLM(config)
    return model, torch.optim.AdamW(model.parameters(), lr=0.004)

def train_step(model, optimizer, batch):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    output = model(batch, labels=batch)
    loss = output.loss + 0.01 * output.auxiliary_loss
    assert torch.isfinite(loss).item()
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0, error_if_nonfinite=True)
    assert grad_norm.item() > 0
    optimizer.step()
    return loss.detach().item()

@torch.inference_mode()
def evaluate(model, ids):
    model.eval()
    output = model(ids, labels=ids)
    accuracy = (output.logits[:, :-1].argmax(-1) == ids[:, 1:]).float().mean()
    return {"cross_entropy": output.loss.item(), "next_token_accuracy": accuracy.item()}

In [3]:
report = {}
models = {}
matching_prefix = validation[:, :4]
modified = validation.clone()
modified[:, 4:] = (modified[:, 4:] + 3) % vocabulary_size
for backend in ("pdf_conv", "mamba2_reference"):
    model, optimizer = new_pair(backend)
    before = evaluate(model, validation)
    training = [train_step(model, optimizer, batch) for batch in train_batches]
    after = evaluate(model, validation)
    with torch.inference_mode():
        prefix = model(matching_prefix).logits
        prefix_original = model(validation).logits[:, : matching_prefix.size(1)]
        prefix_changed = model(modified).logits[:, : matching_prefix.size(1)]
        torch.testing.assert_close(prefix, prefix_original, atol=1e-6, rtol=1e-5)
        torch.testing.assert_close(prefix_original, prefix_changed, atol=1e-6, rtol=1e-5)
    report[backend] = {
        "parameters": model.parameter_count(),
        "train_targets": sum(batch.shape[0] * (batch.shape[1] - 1) for batch in train_batches),
        "initial_validation": before,
        "final_validation": after,
        "first_training_loss": training[0],
        "last_training_loss": training[-1],
        "prefix_max_error": (prefix_original - prefix_changed).abs().max().item(),
    }
    assert math.isfinite(after["cross_entropy"])
    models[backend] = model
print(json.dumps(report, indent=2))

{
  "pdf_conv": {
    "parameters": 6992,
    "train_targets": 1024,
    "initial_validation": {
      "cross_entropy": 2.94523286819458,
      "next_token_accuracy": 0.03125
    },
    "final_validation": {
      "cross_entropy": 2.033109426498413,
      "next_token_accuracy": 0.6875
    },
    "first_training_loss": 2.9591548442840576,
    "last_training_loss": 1.906134843826294,
    "prefix_max_error": 0.0
  },
  "mamba2_reference": {
    "parameters": 7248,
    "train_targets": 1024,
    "initial_validation": {
      "cross_entropy": 2.975714921951294,
      "next_token_accuracy": 0.0
    },
    "final_validation": {
      "cross_entropy": 1.722990870475769,
      "next_token_accuracy": 0.96875
    },
    "first_training_loss": 2.9778833389282227,
    "last_training_loss": 1.6353075504302979,
    "prefix_max_error": 0.0
  }
}


## Interrupted training and integrity verification

Save a versioned local snapshot of the native random-initialized model
after two optimization steps. Validate the committed bundle, restore
tensor and Python RNG state, and replay the synthetic input offset.
Compare both loss trajectory and every tensor with a continuous run.

On Windows, the referenced checkpoint implementation may fail when it
flushes a file opened read-only. This notebook reports the native failure
and retries only that fixture with write-capable handles on its own
temporary files. The repository code stays unchanged.

In [4]:
checkpoint_batches = train_batches[:5]
cutoff = 2
baseline, baseline_optimizer = new_pair("mamba2_reference")
continuous = [train_step(baseline, baseline_optimizer, batch) for batch in checkpoint_batches]
run, optimizer = new_pair("mamba2_reference")
prefix_loss = [train_step(run, optimizer, batch) for batch in checkpoint_batches[:cutoff]]
post_cutoff_rng = torch.get_rng_state().clone()
post_cutoff_random = random.getstate()
tokens_seen = cutoff * batch_size * sequence_length

def write_bundle(stage):
    run.save_pretrained(stage / "model")
    torch.save({
        "step": cutoff,
        "tokens_seen": tokens_seen,
        "optimizer": optimizer.state_dict(),
        "torch_rng_state": post_cutoff_rng,
        "python_rng_state": post_cutoff_random,
        "seed": seed,
    }, stage / "training_state.pt")
    (stage / "tokenizer.json").write_text(json.dumps({
        "schema": "synthetic-integer-ids", "vocabulary_size": vocabulary_size,
    }), encoding="utf-8")
    (stage / "run_manifest.json").write_text(json.dumps({
        "seed": seed, "batch_size": batch_size, "sequence_length": sequence_length,
    }), encoding="utf-8")

class WritableCheckpointPath(type(Path())):
    def open(self, mode="r", *args, **kwargs):
        return super().open("r+b" if mode == "rb" else mode, *args, **kwargs)

def commit_fixture(root):
    options = dict(step=cutoff, tokens_seen=tokens_seen, writer=write_bundle)
    try:
        atomic_checkpoint.commit_checkpoint(root, **options)
    except OSError as error:
        if sys.platform != "win32" or error.errno != errno.EBADF:
            raise
        print({"native_windows_checkpoint": "failed", "errno": error.errno,
               "fixture_workaround": "write-capable handles on private temporary files"})
        with patch.object(atomic_checkpoint, "Path", WritableCheckpointPath):
            atomic_checkpoint.commit_checkpoint(root, **options)
    else:
        print({"native_checkpoint": "pass"})

with tempfile.TemporaryDirectory(prefix="ada-preview-") as temporary:
    root = Path(temporary)
    commit_fixture(root)
    committed = atomic_checkpoint.resolve_committed_checkpoint(root)
    assert committed is not None
    resumed = ADAForCausalLM.from_pretrained(committed / "model")
    resumed_optimizer = torch.optim.AdamW(resumed.parameters(), lr=0.004)
    state = torch.load(committed / "training_state.pt", weights_only=True, map_location="cpu")
    resumed_optimizer.load_state_dict(state["optimizer"])
    assert state["tokens_seen"] == tokens_seen
    torch.set_rng_state(state["torch_rng_state"])
    random.setstate(state["python_rng_state"])
    remaining = replay_completed_batches(iter(checkpoint_batches), state["step"])
    resumed_losses = prefix_loss + [train_step(resumed, resumed_optimizer, batch) for batch in remaining]
    torch.testing.assert_close(torch.tensor(resumed_losses), torch.tensor(continuous))
    for name, reference in baseline.state_dict().items():
        torch.testing.assert_close(resumed.state_dict()[name], reference)
print({"resume_status": "pass", "steps_compared": len(continuous), "tokens_at_checkpoint": tokens_seen})

{'native_windows_checkpoint': 'failed', 'errno': 9, 'fixture_workaround': 'write-capable handles on private temporary files'}


{'resume_status': 'pass', 'steps_compared': 5, 'tokens_at_checkpoint': 64}


## Prompt prefill and greedy decoding

ADA's standalone language-model forward takes a full token prefix. This
adapter lets us test the public generator's all-token prefill and one-token
incremental input contract. It recomputes the full prefix in the model on
each call. Track call lengths separately from the model's work.

In [5]:
class FullPrefixRunner:
    def __init__(self, model):
        self.model = model.eval()
        self.call_lengths = []

    def step(self, tokens, state):
        self.call_lengths.append(tokens.shape[1])
        prefix = tokens if state is None else torch.cat((state, tokens), dim=1)
        return self.model(prefix).logits[:, -1:], prefix

model = models["mamba2_reference"]
runner = FullPrefixRunner(model)
prompt = validation[:2, :4]
new_tokens = 4
generated, state = generation.AutoregressiveGenerator(runner).generate(
    prompt, config=generation.GenerationConfig(max_new_tokens=new_tokens, top_k=1)
)
expected = prompt
with torch.inference_mode():
    for _ in range(new_tokens):
        next_id = model(expected).logits[:, -1].argmax(-1, keepdim=True)
        expected = torch.cat((expected, next_id), dim=1)
assert runner.call_lengths == [prompt.size(1)] + [1] * (new_tokens - 1)
torch.testing.assert_close(generated, expected)
print({"generation_status": "pass", "call_lengths": runner.call_lengths, "generated_ids": generated.tolist()})

{'generation_status': 'pass', 'call_lengths': [4, 1, 1, 1], 'generated_ids': [[15, 16, 17, 18, 0, 1, 2, 3], [16, 17, 18, 0, 1, 2, 3, 4]]}


## Research reading map

| Source | What it contributes to a future controlled ADA comparison |
| --- | --- |
| [Attention Is All You Need](https://arxiv.org/abs/1706.03762v7) | Standard scaled dot-product attention and causal masking as a comparison point for hybrid layers. |
| [RoFormer](https://arxiv.org/abs/2104.09864v5) | A position-encoding ablation. ADA's preview LM uses learned position embeddings, so this run makes no RoPE result claim. |
| [MUDDFormer](https://arxiv.org/abs/2502.12170v2) | A separate test of input-dependent cross-layer connections at matched costs. |
| [Pangu Ultra](https://arxiv.org/abs/2504.07866v2) | Depth-scaled normalization and system evidence on Ascend. The local CPU experiment cannot validate the published large-scale result. |
| [SpanNorm](https://arxiv.org/abs/2601.22580v2) | Motivation to measure depth, signal variance and gradients under a normalization change. |
| [Revisiting the Shape Convention](https://arxiv.org/abs/2602.06471v1) | Width/depth and hourglass ablations at matched parameter and context budgets. |
| [An Introduction to Statistical Learning](https://www.statlearning.com/) | Validation design, held-out evaluation and separating exploratory observations from generalization claims. |
| [Mamba-2](https://arxiv.org/abs/2405.21060) | Math reference for the opt-in mixer. Kernel parity, memory and latency require their own measurements. |

The other notebooks in this directory isolate historical or companion
components. They preserve their own source boundaries and identify
reproduced failures and newer controls. Before scaling, run matched-token
experiments over multiple seeds, test longer contexts, add hardware
profiling, and inspect the rules governing data and artifact release.